<a href="https://colab.research.google.com/github/RafaelaMlucca/violence-signatures-sinan/blob/main/notebooks/05_transformacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Transformação para Modelagem

**Projeto:** Assinaturas preditivas dos tipos de violência contra a mulher  
**Autora:** Rafaela Lucca


## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import save_npz

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
df = pd.read_parquet(DRIVE / 'viol_mulher.parquet')
print(f'Base carregada: {df.shape}')

# Limite de cardinalidade para sinalizar tratamento
LIMITE_CARDINALIDADE = 20
TETO_COLUNAS = 5  # máximo de categorias após agrupamento

Base carregada: (3898186, 80)


## 1. Reconstruir variáveis básicas

In [3]:
import gc

# --- Redução de memória: string vazia -> NaN + conversão para category ---
mem_antes = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memória antes: {mem_antes:.0f} MB')

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('', np.nan)
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')
gc.collect()

mem_depois = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memória depois: {mem_depois:.0f} MB ({100*(1-mem_depois/mem_antes):.0f}% de reducao)')

# --- Idade: versao vetorizada (sem .apply, mais leve) ---
def idade_vetorizada(serie):
    s = serie.astype(str).str.zfill(4)
    primeiro = s.str[0]
    resto = pd.to_numeric(s.str[1:], errors='coerce')
    idade = np.where(primeiro == '4', resto,
             np.where(primeiro.isin(['1','2','3']), 0, np.nan))
    return idade.astype('float32')

df['IDADE'] = idade_vetorizada(df['NU_IDADE_N'])
df.loc[df['IDADE'] > 110, 'IDADE'] = np.nan
gc.collect()

# --- Alvos binarios ---
for col in ['VIOL_FISIC', 'VIOL_PSICO', 'VIOL_SEXU']:
    novo = 'y_' + col.replace('VIOL_', '').lower()
    df[novo] = df[col].astype(str).map({'1': 1, '2': 0}).astype('float32')

df['NU_ANO'] = pd.to_numeric(df['NU_ANO'], errors='coerce').astype('Int16')
gc.collect()

# --- Duplicatas: apenas verifica (remocao custa muita RAM p/ efeito minimo) ---
n_dup = df.duplicated().sum()
pct_dup = 100 * n_dup / len(df)
print(f'\nLinhas duplicadas: {n_dup:,} ({pct_dup:.2f}%)')
if pct_dup >= 0.5:
    print('Proporcao relevante — considere remover com df.drop_duplicates().')
else:
    print('Proporcao desprezivel — duplicatas mantidas.')

print(f'\nApos preparacao: {df.shape}')

Memória antes: 13464 MB
Memória depois: 403 MB (97% de reducao)

Linhas duplicadas: 1,678 (0.04%)
Proporcao desprezivel — duplicatas mantidas.

Apos preparacao: (3898186, 81)


## 2. Definição das features de modelagem

In [4]:
ALVOS = ['y_fisic', 'y_psico', 'y_sexu']

EXCLUIR = (
    ALVOS +
    [
        'NU_IDADE_N', 'CS_SEXO',
        'NU_ANO', 'DT_NOTIFIC', 'DT_OCOR', 'ANO_NASC',  # ANO_NASC redundante com IDADE
        'VIOL_FISIC', 'VIOL_PSICO', 'VIOL_SEXU',
        'VIOL_TORT', 'VIOL_TRAF', 'VIOL_FINAN', 'VIOL_NEGLI',
        'VIOL_INFAN', 'VIOL_LEGAL', 'VIOL_OUTR',
        'SEX_ASSEDI', 'SEX_ESTUPR', 'SEX_PORNO', 'SEX_EXPLO', 'SEX_OUTRO',
        'ID_MN_OCOR', 'ID_MN_RESI',  # alta cardinalidade — códigos de município
        'SG_UF_NOT',  # redundante com SG_UF_OCOR (UF de notificação ≈ UF de ocorrência)
    ]
)

features = [c for c in df.columns if c not in EXCLUIR]
NUMERICAS = ['IDADE']
CATEGORICAS = [c for c in features if c not in NUMERICAS]
print(f'Total de features: {len(features)}')
print(f'  Numéricas: {len(NUMERICAS)}')
print(f'  Categóricas: {len(CATEGORICAS)}')

Total de features: 54
  Numéricas: 1
  Categóricas: 53


## 3. Auditoria de cardinalidade

Para cada variável categórica, contamos o número de valores únicos. Variáveis acima do limite serão sinalizadas para tratamento.

In [5]:
auditoria = []
for col in CATEGORICAS:
    n_unico = df[col].nunique(dropna=False)
    auditoria.append({'variavel': col, 'n_categorias': n_unico})

aud = pd.DataFrame(auditoria).sort_values('n_categorias', ascending=False)

def classificar(n):
    if n <= 15:                    return 'OK'
    elif n <= LIMITE_CARDINALIDADE: return 'ATENCAO'
    else:                          return 'PROBLEMA'

aud['situacao'] = aud['n_categorias'].apply(classificar)

print('AUDITORIA DE CARDINALIDADE\n' + '='*55)
print(aud.to_string(index=False))

print('\n' + '='*55)
print('RESUMO:')
print(aud['situacao'].value_counts().to_string())

problematicas = aud[aud['situacao']=='PROBLEMA']['variavel'].tolist()
print(f'\nVariáveis que precisam de tratamento: {problematicas}')

AUDITORIA DE CARDINALIDADE
  variavel  n_categorias situacao
 HORA_OCOR          1443 PROBLEMA
SG_UF_OCOR            29 PROBLEMA
CS_ESCOL_N            13       OK
LOCAL_OCOR            12       OK
CS_GESTANT             8       OK
  CICL_VID             7       OK
SIT_CONJUG             7       OK
   CS_RACA             7       OK
ORIENT_SEX             6       OK
DEF_VISUAL             6       OK
   DEF_OUT             6       OK
 IDENT_GEN             6       OK
DEF_AUDITI             6       OK
   REL_CAT             6       OK
NUM_ENVOLV             6       OK
REL_OUTROS             6       OK
AUTOR_SEXO             6       OK
 TRAN_COMP             6       OK
 TRAN_MENT             6       OK
DEF_FISICA             5       OK
DEF_MENTAL             5       OK
   REL_POL             5       OK
  REL_INST             5       OK
  REL_TRAB             5       OK
   REL_MAD             5       OK
REL_PROPRI             5       OK
 DEF_TRANS             5       OK
   REL_PAD           

## 4. Tratamento das variáveis de alta cardinalidade

Aplicamos agrupamento **não-supervisionado** (sem olhar o alvo). Cada variável recebe a estratégia adequada ao seu tipo.

In [6]:
# --- Estratégia 1: HORA_OCOR -> faixas horárias de domínio ---
def faixa_horaria(h):
    """Agrupa hora (HH:MM) em 4 faixas com sentido epidemiológico."""
    if pd.isna(h) or str(h).strip() in ('', 'MISSING', 'nan'):
        return 'IGNORADO'
    try:
        hora = int(str(h).split(':')[0])
    except (ValueError, IndexError):
        return 'IGNORADO'
    if   0 <= hora < 6:   return '1_madrugada'
    elif 6 <= hora < 12:  return '2_manha'
    elif 12 <= hora < 18: return '3_tarde'
    elif 18 <= hora < 24: return '4_noite'
    return 'IGNORADO'

if 'HORA_OCOR' in df.columns:
    antes = df['HORA_OCOR'].nunique(dropna=False)
    df['HORA_OCOR'] = df['HORA_OCOR'].apply(faixa_horaria)
    depois = df['HORA_OCOR'].nunique(dropna=False)
    print(f'HORA_OCOR: {antes} categorias -> {depois} faixas')
    print(f'  Distribuição: {df["HORA_OCOR"].value_counts().to_dict()}')

HORA_OCOR: 1443 categorias -> 5 faixas
  Distribuição: {'IGNORADO': 1785013, '4_noite': 768750, '3_tarde': 575731, '2_manha': 399282, '1_madrugada': 369410}


In [7]:
# --- Estratégia 3: SG_UF_OCOR -> macrorregiões do Brasil ---
# O código IBGE de UF tem o primeiro dígito indicando a região:
# 1=Norte · 2=Nordeste · 3=Sudeste · 4=Sul · 5=Centro-Oeste
def uf_para_regiao(cod):
    """Agrupa código IBGE de UF em macrorregião. Não-supervisionado."""
    if pd.isna(cod) or str(cod).strip() in ('', 'MISSING', 'nan'):
        return 'IGNORADO'
    s = str(cod).strip()
    # Remove eventual sufixo decimal ('43.0' -> '43')
    s = s.split('.')[0]
    primeiro = s[0] if s else ''
    mapa = {'1':'Norte', '2':'Nordeste', '3':'Sudeste',
            '4':'Sul', '5':'CentroOeste'}
    return mapa.get(primeiro, 'IGNORADO')

if 'SG_UF_OCOR' in df.columns:
    antes = df['SG_UF_OCOR'].nunique(dropna=False)
    # .astype(str) descongela a coluna 'category' (evita erro de setitem)
    df['SG_UF_OCOR'] = df['SG_UF_OCOR'].astype(str).apply(uf_para_regiao)
    depois = df['SG_UF_OCOR'].nunique(dropna=False)
    print(f'SG_UF_OCOR: {antes} categorias -> {depois} macrorregiões')
    print(f'  Distribuição: {df["SG_UF_OCOR"].value_counts().to_dict()}')

# --- Estratégia genérica: top-N + OUTROS (para outras nominais, se houver) ---
def agrupa_top_n(serie, top_n=TETO_COLUNAS-1):
    """Mantém as top_n categorias mais frequentes; agrupa o resto em OUTROS.
    Converte para str antes, evitando erro de inserção em coluna categórica."""
    s = serie.astype(str)
    frequentes = s.value_counts().head(top_n).index.tolist()
    return s.where(s.isin(frequentes), 'OUTROS')

# Variáveis ainda problemáticas, fora as já tratadas
ja_tratadas = {'HORA_OCOR', 'ANO_NASC', 'SG_UF_OCOR', 'SG_UF_NOT'}
nominais_problema = [v for v in problematicas if v not in ja_tratadas]

if nominais_problema:
    print(f'\nOutras nominais para agrupamento top-N: {nominais_problema}')
    for col in nominais_problema:
        antes = df[col].nunique(dropna=False)
        df[col] = agrupa_top_n(df[col])
        depois = df[col].nunique(dropna=False)
        print(f'  {col}: {antes} -> {depois} categorias')
else:
    print('\nNenhuma outra variável nominal precisou de agrupamento top-N.')

SG_UF_OCOR: 29 categorias -> 6 macrorregiões
  Distribuição: {'IGNORADO': 1717426, 'Sudeste': 1059322, 'Sul': 417335, 'Nordeste': 388533, 'CentroOeste': 173488, 'Norte': 142082}

Nenhuma outra variável nominal precisou de agrupamento top-N.


In [8]:
# Verificação pós-tratamento
print('VERIFICAÇÃO — cardinalidade após tratamento\n' + '='*55)
aud_pos = []
for col in CATEGORICAS:
    aud_pos.append({'variavel': col, 'n_categorias': df[col].nunique(dropna=False)})
aud_pos = pd.DataFrame(aud_pos).sort_values('n_categorias', ascending=False)
print(aud_pos.head(15).to_string(index=False))

ainda_problema = aud_pos[aud_pos['n_categorias'] > LIMITE_CARDINALIDADE]
if len(ainda_problema) == 0:
    print(f'\n✓ Todas as variáveis estão abaixo de {LIMITE_CARDINALIDADE} categorias.')
else:
    print(f'\n⚠ Ainda há variáveis acima do limite: {ainda_problema["variavel"].tolist()}')

VERIFICAÇÃO — cardinalidade após tratamento
  variavel  n_categorias
CS_ESCOL_N            13
LOCAL_OCOR            12
CS_GESTANT             8
SIT_CONJUG             7
   CS_RACA             7
  CICL_VID             7
SG_UF_OCOR             6
 IDENT_GEN             6
ORIENT_SEX             6
DEF_VISUAL             6
   DEF_OUT             6
 TRAN_COMP             6
DEF_AUDITI             6
REL_OUTROS             6
   REL_CAT             6

✓ Todas as variáveis estão abaixo de 20 categorias.


## 5. Split temporal

Treino: 2018-2023 · Teste: 2024.

In [9]:
# Split temporal usando NU_ANO (preservado no df, fora de 'features')
train = df[df['NU_ANO'] <= 2023].copy()
test  = df[df['NU_ANO'] == 2024].copy()
print(f'Treino (2018-2023): {len(train):>9,} ({100*len(train)/len(df):.1f}%)')
print(f'Teste  (2024):      {len(test):>9,} ({100*len(test)/len(df):.1f}%)')

# Verificação de integridade
assert len(train) > 0 and len(test) > 0, 'Split falhou — verifique NU_ANO'
print('\\n✓ Split temporal realizado com sucesso.')

Treino (2018-2023): 2,556,340 (65.6%)
Teste  (2024):        875,656 (22.5%)
\n✓ Split temporal realizado com sucesso.


## 6. Pipeline de transformação

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])
preprocessor = ColumnTransformer([
    ('num', pipe_num, NUMERICAS),
    ('cat', pipe_cat, CATEGORICAS)
])

X_train = train[features]
X_test  = test[features]

print('Ajustando pipeline no treino...')
X_train_t = preprocessor.fit_transform(X_train)
print(f'  X_train: {X_train_t.shape}')

X_test_t = preprocessor.transform(X_test)
print(f'  X_test:  {X_test_t.shape}')

print(f'\nDimensionalidade após One-Hot: {X_train_t.shape[1]} colunas')
print('(Compare com a versão anterior: ~1764 colunas. A redução')
print(' reflete o agrupamento das variáveis de alta cardinalidade.)')

Ajustando pipeline no treino...
  X_train: (2556340, 275)
  X_test:  (875656, 275)

Dimensionalidade após One-Hot: 275 colunas
(Compare com a versão anterior: ~1764 colunas. A redução
 reflete o agrupamento das variáveis de alta cardinalidade.)


## 7. Distribuição dos alvos e salvamento

In [11]:
print('Distribuição dos alvos:\n')
for alvo in ALVOS:
    n_tr = (train[alvo]==1).sum()
    p_tr = 100*n_tr/train[alvo].notna().sum()
    n_te = (test[alvo]==1).sum()
    p_te = 100*n_te/test[alvo].notna().sum()
    print(f'  {alvo}: treino {p_tr:.1f}% | teste {p_te:.1f}%')

Distribuição dos alvos:

  y_fisic: treino 51.4% | teste 47.6%
  y_psico: treino 23.8% | teste 22.1%
  y_sexu: treino 16.1% | teste 17.2%


In [12]:
joblib.dump(preprocessor, DRIVE / 'preprocessor.joblib')
save_npz(DRIVE / 'X_train.npz', X_train_t)
save_npz(DRIVE / 'X_test.npz',  X_test_t)
train[ALVOS].to_parquet(DRIVE / 'y_train.parquet', index=False)
test[ALVOS].to_parquet(DRIVE / 'y_test.parquet', index=False)

feature_names = preprocessor.get_feature_names_out().tolist()
with open(DRIVE / 'feature_names.json', 'w') as f:
    json.dump(feature_names, f)

# Salva também a auditoria de cardinalidade
aud.to_csv(DRIVE / 'auditoria_cardinalidade.csv', index=False)

print(f'Arquivos salvos em {DRIVE}:')
print(f'  preprocessor.joblib')
print(f'  X_train.npz · {X_train_t.shape}')
print(f'  X_test.npz  · {X_test_t.shape}')
print(f'  y_train.parquet · y_test.parquet')
print(f'  feature_names.json · {len(feature_names)} nomes')
print(f'  auditoria_cardinalidade.csv')

Arquivos salvos em /content/drive/MyDrive/projeto_violencia_mulher:
  preprocessor.joblib
  X_train.npz · (2556340, 275)
  X_test.npz  · (875656, 275)
  y_train.parquet · y_test.parquet
  feature_names.json · 275 nomes
  auditoria_cardinalidade.csv
